##rag pipeline form ingestion to embedding to storing in vector db

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [ ]:
def process_all_pdfs(pdf_directory):
    """prcess all pdf files in the specified directory and return a list of documents"""
    
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    #find all the pdfs in the directory
    pdf_files = list(pdf_dir.glob("*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files in the directory.")
    
    
    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}...")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            
            #add the source of information to each document
            for doc in documents:
                doc.metadata['source'] = str(pdf_file.name)
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f" Successfully processed {pdf_file.name}.\n")
            
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
            
    print(f"Total documents processed: {len(all_documents)}")
    
    return all_documents
                

all_pdfs = process_all_pdfs("../data/datafiles")


Found 3 PDF files in the directory.
Processing Letter of Gratitude .pdf...
Successfully processed Letter of Gratitude .pdf.

Processing TA_Database_SQL_NoSQL_Assignment_Proposal.pdf...
Successfully processed TA_Database_SQL_NoSQL_Assignment_Proposal.pdf.

Processing project_idea_iris_detection_abdul-kalam.pdf...
Successfully processed project_idea_iris_detection_abdul-kalam.pdf.

Total documents processed: 4


In [16]:
all_pdfs

[Document(metadata={'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'Letter of Gratitude .pdf', 'file_path': '../data/datafiles/Letter of Gratitude .pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'Letter of Gratitude', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'file_type': 'pdf'}, page_content='Letter of Gratitude for Educational Support \n \nDear Sir/Madam, \nI hope you are doing well. \nI am writing this letter with sincere gratitude and deep appreciation for your \ngenerous support in sponsoring my educational expenses. I am truly thankful for \nyour kindness, which has made it possible for me to continue my studies at FAST \nNational University of Computer and Emerging Sciences without financial stress \nand to stay focused on achieving my academic and life goals. \nThis support has had a life-changing impact on me. It has not only eased a great \nbur

In [26]:
#chunking the data into smaller pieces for better processing


def split_documents(docs ,chunk_size, chunk_overlap):
    """split the docs in to smaller chunks"""
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
        )
    
    split_docs = text_splitter.split_documents(docs)
    print(f"split {len(docs)} documents into Total chunks created: {len(split_docs)}")
    
    
    #show the example chunk
    if split_docs:
        print(f"Example chunk:\n")
        print(f"content: {split_docs[0].page_content[:200]}...")
        print(f"metadata: {split_docs[0].metadata}")
        
    return split_docs


In [27]:
#function call

chunks =split_documents(all_pdfs, chunk_size=250, chunk_overlap=50)
chunks

split 4 documents into Total chunks created: 29
Example chunk:

content: Letter of Gratitude for Educational Support 
 
Dear Sir/Madam, 
I hope you are doing well. 
I am writing this letter with sincere gratitude and deep appreciation for your...
metadata: {'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'Letter of Gratitude .pdf', 'file_path': '../data/datafiles/Letter of Gratitude .pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'Letter of Gratitude', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'Letter of Gratitude .pdf', 'file_path': '../data/datafiles/Letter of Gratitude .pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'Letter of Gratitude', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'file_type': 'pdf'}, page_content='Letter of Gratitude for Educational Support \n \nDear Sir/Madam, \nI hope you are doing well. \nI am writing this letter with sincere gratitude and deep appreciation for your'),
 Document(metadata={'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'Letter of Gratitude .pdf', 'file_path': '../data/datafiles/Letter of Gratitude .pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'Letter of Gratitude', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0,